# SO101 ACT Training v3
Internet + GPU T4 required. HF_TOKEN via Kaggle Secret.

In [ ]:
!curl -s https://ipinfo.io 2>/dev/null && echo 'INTERNET OK' || echo 'INTERNET FAILED'

In [ ]:
import os
os.environ.pop('HF_ENDPOINT', None)
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

DATASET_REPO = 'lerobot/svla_so101_pickplace'
MODEL_REPO = 'xieyucheng123/so101-act'
TRAINING_STEPS = 10000
print(f'Config ready, token len={len(HF_TOKEN)}')

In [ ]:
!pip install -q 'lerobot[dataset]' 2>&1 | tail -15

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import lerobot
print(f'lerobot version: {getattr(lerobot, "__version__", "unknown")}')
from lerobot.datasets.lerobot_dataset import LeRobotDataset
print('LeRobotDataset imported OK')

In [ ]:
from huggingface_hub import HfApi
print(f'Using dataset: {DATASET_REPO} (will stream from HF Hub during training)')

In [ ]:
dataset = LeRobotDataset(DATASET_REPO)
print(f'Episodes: {dataset.num_episodes}')
print(f'Frames: {dataset.num_frames}')

In [ ]:
import subprocess
result = subprocess.run(['lerobot-train', '--help'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if 'repo' in line.lower() or 'hub' in line.lower() or 'output' in line.lower() or 'save' in line.lower():
        print(line)

In [ ]:
import subprocess
cmd = [
    'lerobot-train',
    '--policy.type=act',
    f'--policy.repo_id={MODEL_REPO}',
    f'--dataset.repo_id={DATASET_REPO}',
    '--batch_size=16',
    f'--steps={TRAINING_STEPS}',
    '--save_freq=5000',
    '--log_freq=100',
    '--output_dir=/kaggle/working/outputs/train/so101_act',
]
print(f'Running: {" ".join(cmd)}')
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-3000:] if result.stdout else 'No stdout')
if result.returncode != 0:
    print(f'STDERR (last 2000): {result.stderr[-2000:]}')
else:
    print('Training complete!')

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
try:
    api.create_repo(repo_id=MODEL_REPO, repo_type='model', exist_ok=True)
except Exception as e:
    print(f'Repo: {e}')

api.upload_folder(
    folder_path='/kaggle/working/outputs/train/so101_act',
    repo_id=MODEL_REPO,
    repo_type='model',
    token=HF_TOKEN,
)
print(f'Model uploaded to {MODEL_REPO}')